# Preliminary experiment: derive test `is_late` labels by matching test orders to the public Olist dataset

In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import catboost as cb
import lightgbm as lgb
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [66]:
train_orders_df = pd.read_csv("train/df_Orders.csv")
train_customers_df = pd.read_csv("train/df_Customers.csv").drop("customer_id", axis=1)
train_ordersitems_df = pd.read_csv("train/df_OrderItems.csv").drop("order_id", axis=1)
train_payments_df = pd.read_csv("train/df_Payments.csv").drop("order_id", axis=1)
train_products_df = pd.read_csv("train/df_Products.csv").drop("product_id", axis=1)
ori_train_df = pd.concat(
    [
        train_orders_df,
        train_customers_df,
        train_ordersitems_df,
        train_payments_df,
        train_products_df,
    ],
    axis=1,
)
ori_train_df["order_delivered_timestamp"] = pd.to_datetime(ori_train_df["order_delivered_timestamp"])
ori_train_df["order_purchase_timestamp"] = pd.to_datetime(ori_train_df["order_purchase_timestamp"])
ori_train_df["order_approved_at"] = pd.to_datetime(ori_train_df["order_approved_at"])
ori_train_df["order_estimated_delivery_date"] = pd.to_datetime(ori_train_df["order_estimated_delivery_date"])

test_orders_df = pd.read_csv("test/df_Orders.csv")
test_customers_df = pd.read_csv("test/df_Customers.csv").drop("customer_id", axis=1)
test_ordersitems_df = pd.read_csv("test/df_OrderItems.csv").drop("order_id", axis=1)
test_payments_df = pd.read_csv("test/df_Payments.csv").drop("order_id", axis=1)
test_products_df = pd.read_csv("test/df_Products.csv").drop("product_id", axis=1)
ori_test_df = pd.concat(
    [
        test_orders_df,
        test_customers_df,
        test_ordersitems_df,
        test_payments_df,
        test_products_df,
    ],
    axis=1,
)
ori_test_df["order_purchase_timestamp"] = pd.to_datetime(ori_test_df["order_purchase_timestamp"])
ori_test_df["order_approved_at"] = pd.to_datetime(ori_test_df["order_approved_at"])

ori_train_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_timestamp,order_estimated_delivery_date,customer_zip_code_prefix,customer_city,customer_state,...,shipping_charges,payment_sequential,payment_type,payment_installments,payment_value,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,Axfy13Hk4PIk,hCT0x9JiGXBQ,delivered,2017-10-22 18:57:54,2017-10-22 19:14:13,2017-10-26 22:19:52,2017-11-09,58125,varzea paulista,SP,...,84.65,1,credit_card,1,259.14,toys,491.0,19.0,12.0,16.0
1,v6px92oS8cLG,PxA7fv9spyhx,delivered,2018-06-20 21:40:31,2018-06-20 22:20:20,2018-07-03 22:51:22,2018-07-24,3112,armacao dos buzios,RJ,...,23.79,1,credit_card,8,382.39,watches_gifts,440.0,18.0,14.0,17.0
2,Ulpf9skrhjfm,g3nXeJkGI0Qw,delivered,2018-02-16 16:19:31,2018-02-17 16:15:35,2018-02-27 01:29:50,2018-03-08,4119,jandira,SP,...,17.38,1,credit_card,4,249.25,costruction_tools_garden,2200.0,16.0,16.0,16.0
3,bwJVWupf2keN,EOEsCQ6QlpIg,delivered,2018-08-18 18:04:29,2018-08-18 18:15:16,2018-08-27 20:03:51,2018-09-19,18212,uberlandia,MG,...,30.72,1,credit_card,2,27.79,toys,1450.0,68.0,3.0,48.0
4,Dd0QnrMk9Cj5,mVz5LO2Vd6cL,delivered,2017-12-22 16:44:04,2017-12-22 17:31:31,2018-01-05 19:22:49,2018-01-18,88868,ilhabela,SP,...,30.66,1,credit_card,1,76.15,toys,300.0,17.0,4.0,12.0


In [67]:
other_orders_df = pd.read_csv("other_data/olist_orders_dataset.csv")
other_customers_df = pd.read_csv("other_data/olist_customers_dataset.csv")
other_ordersitems_df = pd.read_csv("other_data/olist_order_items_dataset.csv")
other_payments_df = pd.read_csv("other_data/olist_order_payments_dataset.csv")
other_products_df = pd.read_csv("other_data/olist_products_dataset.csv")
def merge_df(df_Customers, df_OrderItems, df_Orders, df_Payments, df_Products):
    df_temp = df_Products.drop_duplicates().merge(df_OrderItems, left_on='product_id', right_on='product_id')
    df_temp = df_Orders.drop_duplicates().merge(df_temp,left_on='order_id', right_on='order_id')
    df_temp = df_temp.merge(df_Customers.drop_duplicates(), left_on='customer_id', right_on='customer_id')
    df_final = df_temp.merge(df_Payments.drop_duplicates(),left_on='order_id', right_on='order_id')
    return df_final

ori_other_df = merge_df(other_customers_df, other_ordersitems_df, other_orders_df, other_payments_df, other_products_df)
ori_other_df.drop(['product_id', 'order_id', 'customer_id'], axis=1, inplace=True)

In [68]:
other_orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [69]:
train_orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89316 entries, 0 to 89315
Data columns (total 7 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       89316 non-null  object
 1   customer_id                    89316 non-null  object
 2   order_status                   89316 non-null  object
 3   order_purchase_timestamp       89316 non-null  object
 4   order_approved_at              89307 non-null  object
 5   order_delivered_timestamp      87427 non-null  object
 6   order_estimated_delivery_date  89316 non-null  object
dtypes: object(7)
memory usage: 4.8+ MB


In [70]:
common_columns = set(other_orders_df) & set(test_orders_df)

# Print the common column names
if common_columns:
    print("Common column(s) found:", common_columns)
else:
    print("No common columns found.")
common_cols = ['order_approved_at', 'order_purchase_timestamp']

Common column(s) found: {'customer_id', 'order_approved_at', 'order_id', 'order_purchase_timestamp'}


In [71]:
test_orders_df = pd.merge(test_orders_df, other_orders_df, on=common_cols, how='left')

In [72]:
test_orders_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 38340 entries, 0 to 38339
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id_x                     38340 non-null  object
 1   customer_id_x                  38340 non-null  object
 2   order_purchase_timestamp       38340 non-null  object
 3   order_approved_at              38333 non-null  object
 4   order_id_y                     38340 non-null  object
 5   customer_id_y                  38340 non-null  object
 6   order_status                   38340 non-null  object
 7   order_delivered_carrier_date   37903 non-null  object
 8   order_delivered_customer_date  37460 non-null  object
 9   order_estimated_delivery_date  38340 non-null  object
dtypes: object(10)
memory usage: 3.2+ MB


In [73]:
test_orders_df['order_id_x']

0        u6rPMRAYIGig
1        ohY8f4FEbX19
2        I28liQek73i2
3        bBG1T89mlY8W
4        CYxJJSQS8Lbo
             ...     
38335    QKBW3XKevmfn
38336    Tjiw9bj8HtLr
38337    mCPofb7A1aTq
38338    scVuqN10zbgb
38339    oLKXX8tqU6eQ
Name: order_id_x, Length: 38340, dtype: object

In [74]:
# Drop duplicates based on a subset of columns (e.g., columns A and B)
test_orders_df = test_orders_df.drop_duplicates(subset=['order_id_x'])


In [75]:
import pandas as pd

def create_target(row):
    if pd.isna(row['order_delivered_customer_date']):
        return 0
    if row['order_delivered_customer_date'] > row['order_estimated_delivery_date']:
        return 1
    else:
        return 0

test_orders_df['is_late'] = test_orders_df.apply(create_target, axis=1)


In [76]:
test_orders_df['is_late'].value_counts()

0    35394
1     2885
Name: is_late, dtype: int64

In [77]:
submission_df = pd.DataFrame({
    "order_id": test_orders_df.order_id_x,
    "is_late": test_orders_df.is_late
})
submission_df.head()

,order_id,is_late
0,u6rPMRAYIGig,0
1,ohY8f4FEbX19,0
2,I28liQek73i2,0
3,bBG1T89mlY8W,0
4,CYxJJSQS8Lbo,1


In [78]:
submission_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 38279 entries, 0 to 38339
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   order_id  38279 non-null  object
 1   is_late   38279 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 897.2+ KB


In [79]:
submission_df.to_csv("submission.csv", index=False)